# 02 — SAX et iterparse — streaming XML

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre les différences entre DOM, SAX et iterparse ;
- utiliser `xml.sax` pour parser un XML en streaming ;
- écrire un `ContentHandler` avec gestion d’état ;
- utiliser `lxml.etree.iterparse` pour libérer la mémoire ;
- parser des fichiers XML de plusieurs Go sans charger en RAM ;
- calculer des statistiques en streaming.

## Prérequis — ce que vous devez maîtriser

À ce stade de la formation, vous savez :

- bases XML et parsing ;
- classes Python (pour les handlers SAX).

Ce que ce notebook **ne couvrira pas** :

- XSD et validation (voir `01_Xml_schemas.ipynb`) ;
- XPath et XSLT (voir `03_Xpath_et_xslt.ipynb`).

## Plan

1. DOM vs SAX vs iterparse
2. SAX — `parseString` et `ContentHandler`
3. ContentHandler avec état
4. Filtrage SAX
5. `iterparse` avec lxml
6. Libération mémoire
7. Statistiques en streaming
8. Performances comparées
9. Synthèse
10. Exercices
11. Ressources externes

### Exemple : extraction de données depuis un gros XML

Simulons le parsing d'un fichier XML volumineux (type export de base de données)
en ne gardant en mémoire qu'un élément à la fois.

In [ ]:
from lxml import etree
from io import BytesIO

# Simulation d'un gros XML
xml_data = '<commandes>\n'
for i in range(1000):
    montant = 10 + (i * 7 % 490)
    xml_data += f'  <commande id="{i}" statut="{"livree" if i % 3 else "annulee"}"><montant>{montant}</montant></commande>\n'
xml_data += '</commandes>'

# Parsing en streaming
total_livrees = 0
count_livrees = 0
for event, elem in etree.iterparse(BytesIO(xml_data.encode()), events=('end',)):
    if elem.tag == 'commande' and elem.get('statut') == 'livree':
        total_livrees += float(elem.find('montant').text)
        count_livrees += 1
    elem.clear()  # Libération mémoire

print(f'Commandes livrées : {count_livrees}')
print(f'Montant total : {total_livrees:.2f} EUR')
print(f'Panier moyen : {total_livrees/count_livrees:.2f} EUR')

### Exercice : SAX handler avec compteurs multiples

Créez un `ContentHandler` SAX qui, en parcourant un XML de type :
```xml
<ecole>
  <classe niveau="CP"><eleve note="15"/>...</classe>
  <classe niveau="CE1">...</classe>
</ecole>
```
calcule la moyenne des notes **par niveau** sans jamais charger tout le document.

In [ ]:
import xml.sax
from xml.sax.handler import ContentHandler
from io import StringIO

class NotesHandler(ContentHandler):
    def __init__(self):
        self.current_niveau = None
        self.stats = {}  # niveau -> [somme, count]

    def startElement(self, name, attrs):
        if name == 'classe':
            self.current_niveau = attrs.get('niveau', '?')
            if self.current_niveau not in self.stats:
                self.stats[self.current_niveau] = [0.0, 0]
        elif name == 'eleve' and self.current_niveau:
            note = float(attrs.get('note', 0))
            self.stats[self.current_niveau][0] += note
            self.stats[self.current_niveau][1] += 1

# Test
xml_ecole = '<ecole>'
import random; random.seed(42)
for niveau in ['CP', 'CE1', 'CE2']:
    xml_ecole += f'<classe niveau="{niveau}">'
    for _ in range(20):
        xml_ecole += f'<eleve note="{random.randint(5, 20)}"/>'
    xml_ecole += '</classe>'
xml_ecole += '</ecole>'

handler = NotesHandler()
xml.sax.parseString(xml_ecole, handler)
for niv, (s, c) in sorted(handler.stats.items()):
    print(f'{niv} : moyenne = {s/c:.1f} sur {c} élèves')

### Exercice avancé : convertisseur XML-to-CSV en streaming

Écrivez un script qui utilise `iterparse` pour convertir un XML de commandes
en un fichier CSV, ligne par ligne, sans jamais charger tout le XML en mémoire.

Structure d'entrée :
```xml
<commandes>
  <commande id="1"><client>Alice</client><montant>150.00</montant><date>2025-01-15</date></commande>
  ...
</commandes>
```

In [ ]:
from lxml import etree
from io import BytesIO, StringIO
import csv

# Génération XML test
xml_src = '<commandes>\n'
noms = ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve']
for i in range(50):
    xml_src += f'  <commande id="{i+1}"><client>{noms[i%5]}</client><montant>{50+i*3.5:.2f}</montant><date>2025-{1+i%12:02d}-{1+i%28:02d}</date></commande>\n'
xml_src += '</commandes>'

# Conversion streaming
output = StringIO()
writer = csv.writer(output)
writer.writerow(['id', 'client', 'montant', 'date'])

for event, elem in etree.iterparse(BytesIO(xml_src.encode()), events=('end',)):
    if elem.tag == 'commande':
        writer.writerow([
            elem.get('id'),
            elem.findtext('client'),
            elem.findtext('montant'),
            elem.findtext('date'),
        ])
        elem.clear()

print(output.getvalue()[:500])

---

## 1. DOM vs SAX vs iterparse

| Approche | Mémoire | Vitesse | API | Usage |
|---|---|---|---|---|
| **DOM** | Tout en mémoire | Rapide | Arbre complet | Petits fichiers, XPath |
| **SAX** | Constante (streaming) | Variable | Événements (callbacks) | Gros fichiers |
| **iterparse** (lxml) | Streaming + partiel DOM | Rapide | Itérateur + éléments | Meilleur compromis |

### Quand utiliser quoi ?

- **< 100 Mo** : DOM suffit (`etree.parse`)
- **100 Mo – 10 Go** : iterparse avec `clear()`
- **> 10 Go ou mémoire limitée** : SAX pur

---

## 2. SAX — `parseString` et `ContentHandler`

In [ ]:
import xml.sax

class SimpleHandler(xml.sax.ContentHandler):
    def startElement(self, name, attrs):
        print(f"start: {name}, attrs: {dict(attrs)}")
    def endElement(self, name):
        print(f"end  : {name}")
    def characters(self, content):
        if content.strip():
            print(f"text : {content.strip()!r}")

xml.sax.parseString(b'<root><item>Hello</item><item attr="x">World</item></root>', SimpleHandler())

---

## 3. ContentHandler avec état

Pour collecter des données, il faut un handler qui maintient un état.

In [ ]:
class Collector(xml.sax.ContentHandler):
    def __init__(self) -> None:
        self.current: str = ''
        self.items: list[str] = []

    def startElement(self, name, attrs):
        self.current = ''

    def characters(self, content):
        self.current += content

    def endElement(self, name):
        if name == 'item':
            self.items.append(self.current.strip())

h = Collector()
xml.sax.parseString(b'<root><item>Alpha</item><item>Beta</item><item>Gamma</item></root>', h)
print(h.items)

### Attention à `characters()`

Le parseur SAX peut appeler `characters()` **plusieurs fois** pour un même élément. C’est pourquoi on accumule avec `+=`.

---

## 4. Filtrage SAX

In [ ]:
class LargeValues(xml.sax.ContentHandler):
    def __init__(self) -> None:
        self.result: list[int] = []
        self.current = ''

    def startElement(self, name, attrs):
        self.current = ''

    def characters(self, content):
        self.current += content

    def endElement(self, name):
        if name == 'valeur':
            try:
                v = int(self.current)
                if v > 100:
                    self.result.append(v)
            except ValueError:
                pass

h = LargeValues()
xml.sax.parseString(b'<r><valeur>50</valeur><valeur>200</valeur><valeur>150</valeur><valeur>30</valeur></r>', h)
print(f"Valeurs > 100: {h.result}")

In [ ]:
# Filtrage par attribut
class BookFilter(xml.sax.ContentHandler):
    def __init__(self) -> None:
        self.ids: list[str] = []

    def startElement(self, name, attrs):
        if name == 'item' and attrs.get('type') == 'book':
            self.ids.append(attrs.get('id', '?'))

h = BookFilter()
xml.sax.parseString(b"<r><item id='1' type='book'/><item id='2' type='movie'/><item id='3' type='book'/></r>", h)
print(f"Books: {h.ids}")

---

## 5. `iterparse` avec lxml

Combine le streaming de SAX avec l’accès DOM aux éléments.

In [ ]:
from lxml import etree
import io

data = io.BytesIO(b'<root>' + b'<item>val</item>' * 10 + b'</root>')
for event, elem in etree.iterparse(data, events=('end',), tag='item'):
    print(f"{elem.tag}: {elem.text}")
    elem.clear()  # libérer la mémoire

In [ ]:
# Événements start et end
data = io.BytesIO(b'<r><a><b>1</b></a><a><b>2</b></a></r>')
for event, elem in etree.iterparse(data, events=('start', 'end')):
    print(f"{event:5s} {elem.tag}")

---

## 6. Libération mémoire

Pour les gros fichiers, il faut libérer les éléments déjà traités.

In [ ]:
def process_big_xml(path: str) -> int:
    """Compte les éléments <record> dans un gros fichier."""
    count = 0
    for event, elem in etree.iterparse(path, events=('end',), tag='record'):
        count += 1
        # Traiter elem.text, elem.attrib, etc.
        elem.clear()
        # Libérer aussi les ancêtres déjà traités
        while elem.getprevious() is not None:
            del elem.getparent()[0]
    return count

print('Pattern standard pour fichiers de plusieurs Go')

### Pourquoi `getprevious()` + `del` ?

Sans cette étape, les éléments frères précédents restent en mémoire car l’arbre parent les référence toujours.

---

## 7. Statistiques en streaming

In [ ]:
xml_data = io.BytesIO(b'<shop><item price="10"/><item price="25"/><item price="5"/><item price="40"/></shop>')
total = 0.0
count = 0
for event, elem in etree.iterparse(xml_data, events=('end',), tag='item'):
    total += float(elem.get('price', 0))
    count += 1
    elem.clear()
print(f"{count} items, total={total}, moyenne={total/count:.2f}")

In [ ]:
# Agrégation par catégorie
from collections import defaultdict

xml_sales = io.BytesIO(
    b'<ventes>'
    b'<sale><produit>A</produit><montant>10</montant></sale>'
    b'<sale><produit>B</produit><montant>20</montant></sale>'
    b'<sale><produit>A</produit><montant>5</montant></sale>'
    b'<sale><produit>B</produit><montant>15</montant></sale>'
    b'</ventes>'
)
totals: dict[str, float] = defaultdict(float)
for event, elem in etree.iterparse(xml_sales, events=('end',), tag='sale'):
    produit = elem.findtext('produit')
    montant = float(elem.findtext('montant', '0'))
    totals[produit] += montant
    elem.clear()
print(dict(totals))

---

## 8. Performances comparées

| Approche | 10 Mo | 100 Mo | 1 Go | RAM |
|---|---|---|---|---|
| DOM (`etree.parse`) | 0.1 s | 1 s | 10 s | ~ taille fichier |
| SAX (`xml.sax`) | 0.2 s | 2 s | 20 s | ~ constante |
| iterparse + clear | 0.1 s | 1.2 s | 12 s | ~ constante |

*Valeurs indicatives sur un PC standard.*

---

## Synthèse

| Approche | API | Usage |
|---|---|---|
| DOM | `etree.parse(f)` | Petit fichier, XPath |
| SAX | `xml.sax.parse(f, handler)` | Très gros, callbacks |
| iterparse | `etree.iterparse(f, tag=)` | Gros + accès DOM partiel |

### Règles à retenir

- **DOM** : simple mais RAM = taille du fichier.
- **SAX** : mémoire constante mais verbeux (callbacks).
- **iterparse + `clear()`** : meilleur compromis pour les gros fichiers.
- **`while getprevious(): del getparent()[0]`** : libérer les ancêtres.
- **`characters()`** peut être appelé plusieurs fois par élément.

---

---

## Parsing de fichiers reels

### Pattern avec fichier temporaire

In [ ]:
import tempfile, os
from lxml import etree

# Creer un fichier XML temporaire
with tempfile.NamedTemporaryFile(mode='w', suffix='.xml', delete=False) as f:
    f.write('<data>')
    for i in range(100):
        f.write(f'<item id="{i}"><value>{i*i}</value></item>')
    f.write('</data>')
    tmp_path = f.name

# Parser en streaming
count = 0
for _, elem in etree.iterparse(tmp_path, events=('end',), tag='item'):
    count += 1
    elem.clear()

print(f"Parse {count} items")
os.unlink(tmp_path)

---

## ErrorHandler SAX

In [ ]:
import xml.sax

class MyErrorHandler(xml.sax.handler.ErrorHandler):
    def error(self, exception):
        print(f'Erreur: {exception}')
    def fatalError(self, exception):
        print(f'Fatal: {exception}')
    def warning(self, exception):
        print(f'Warning: {exception}')

print('ErrorHandler: error(), fatalError(), warning()')

---

## iterparse avec events start

In [ ]:
from lxml import etree
import io

# Utiliser start pour lire les attributs avant le contenu
data = io.BytesIO(b'<r><item id="1" type="a">X</item><item id="2" type="b">Y</item></r>')
for event, elem in etree.iterparse(data, events=('start', 'end')):
    if event == 'start' and elem.tag == 'item':
        print(f"Start: id={elem.get('id')}, type={elem.get('type')}")
    if event == 'end' and elem.tag == 'item':
        print(f"End: text={elem.text}")
        elem.clear()

---

## Combinaison iterparse + XPath local

In [ ]:
from lxml import etree
import io

xml = io.BytesIO(
    b'<library>'
    b'<book><title>A</title><author>X</author><year>2020</year></book>'
    b'<book><title>B</title><author>Y</author><year>2021</year></book>'
    b'<book><title>C</title><author>X</author><year>2022</year></book>'
    b'</library>'
)
results = []
for _, elem in etree.iterparse(xml, events=('end',), tag='book'):
    # XPath local sur l'element
    author = elem.findtext('author')
    if author == 'X':
        results.append(elem.findtext('title'))
    elem.clear()
print(f"Livres de X: {results}")

---

## Ecriture XML en streaming

In [ ]:
from lxml import etree
import io

# Ecrire un gros XML sans tout en memoire
buf = io.BytesIO()
with etree.xmlfile(buf, encoding='utf-8') as xf:
    with xf.element('root'):
        for i in range(5):
            with xf.element('item', attrib={'id': str(i)}):
                xf.write(f'Valeur {i}')

print(buf.getvalue().decode()[:200])

---

## SAX avec xml.sax.make_parser

In [ ]:
import xml.sax

# Creer un parser configurable
parser = xml.sax.make_parser()

class NameCollector(xml.sax.ContentHandler):
    def __init__(self):
        self.names = []
        self.current = ''
    def startElement(self, name, attrs):
        self.current = ''
    def characters(self, content):
        self.current += content
    def endElement(self, name):
        if name == 'name':
            self.names.append(self.current.strip())

h = NameCollector()
parser.setContentHandler(h)
from io import BytesIO
parser.parse(BytesIO(b'<r><name>Alice</name><name>Bob</name></r>'))
print(h.names)

---

## Mesurer la memoire

In [ ]:
import sys
from lxml import etree
import io

# DOM : tout en memoire
xml_str = b'<r>' + b'<item>x</item>' * 1000 + b'</r>'
doc = etree.fromstring(xml_str)
print(f"DOM: ~{sys.getsizeof(etree.tostring(doc))} octets pour le XML serialise")

In [ ]:
# iterparse : memoire constante (approximation)
data = io.BytesIO(xml_str)
count = 0
for _, elem in etree.iterparse(data, events=('end',), tag='item'):
    count += 1
    elem.clear()
print(f"iterparse: {count} items traites sans tout charger")

---

## Gestion des encodages

In [ ]:
# XML avec declaration d'encodage
xml_latin = b"<?xml version='1.0' encoding='ISO-8859-1'?><r><t>caf\xe9</t></r>"
from lxml import etree
import io
for _, elem in etree.iterparse(io.BytesIO(xml_latin), events=('end',), tag='t'):
    print(f"Texte: {elem.text}")
    elem.clear()

### Encodages courants

| Encodage | Alias | Usage |
|---|---|---|
| UTF-8 | utf-8 | Standard, recommande |
| ISO-8859-1 | latin-1 | Ancien, Europe Ouest |
| Windows-1252 | cp1252 | Ancien, Windows |

---

## Benchmarks SAX vs iterparse

In [ ]:
import time
import xml.sax
from lxml import etree
import io

# Generer un XML de test
xml_bytes = b'<root>' + b'<item>test</item>' * 5000 + b'</root>'

# SAX
class Counter(xml.sax.ContentHandler):
    def __init__(self): self.n = 0
    def startElement(self, name, attrs):
        if name == 'item': self.n += 1

t0 = time.perf_counter()
h = Counter()
xml.sax.parseString(xml_bytes, h)
t_sax = time.perf_counter() - t0

# iterparse
t0 = time.perf_counter()
n = 0
for _, elem in etree.iterparse(io.BytesIO(xml_bytes), events=('end',), tag='item'):
    n += 1
    elem.clear()
t_iter = time.perf_counter() - t0

print(f"SAX:       {h.n} items en {t_sax*1000:.1f} ms")
print(f"iterparse: {n} items en {t_iter*1000:.1f} ms")

## Exercices

### Exercice 1 *(facile)* — Compter les éléments

Avec SAX, comptez le nombre de tags `item` dans un XML.

In [ ]:
import xml.sax

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import xml.sax

class Counter(xml.sax.ContentHandler):
    def __init__(self):
        self.n = 0
    def startElement(self, name, attrs):
        if name == 'item':
            self.n += 1

h = Counter()
xml.sax.parseString(b'<r><item/><item/><item/></r>', h)
print(f"Nombre d'items: {h.n}")
```

</details>

### Exercice 2 *(facile)* — Extraire les titres

Avec iterparse, extrayez tous les textes des éléments `title`.

In [ ]:
from lxml import etree
import io

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree
import io

data = io.BytesIO(b'<catalog><title>Python</title><title>Rust</title><title>Go</title></catalog>')
titles = []
for _, elem in etree.iterparse(data, events=('end',), tag='title'):
    titles.append(elem.text)
    elem.clear()
print(titles)
```

</details>

### Exercice 3 *(moyen)* — Filtrer par attribut (SAX)

Handler SAX qui collecte les `id` des items ayant `type='book'`.

In [ ]:
import xml.sax

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import xml.sax

class BookFilter(xml.sax.ContentHandler):
    def __init__(self):
        self.ids = []
    def startElement(self, name, attrs):
        if name == 'item' and attrs.get('type') == 'book':
            self.ids.append(attrs.get('id'))

h = BookFilter()
xml.sax.parseString(b"<r><item id='1' type='book'/><item id='2' type='movie'/><item id='3' type='book'/></r>", h)
print(f"IDs livres: {h.ids}")
```

</details>

### Exercice 4 *(moyen)* — Somme en streaming

Avec iterparse, calculez la somme des éléments `<n>` dans `<r><n>0</n><n>1</n>...<n>9</n></r>`.

In [ ]:
from lxml import etree
import io

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree
import io

data = io.BytesIO(b'<r>' + b''.join(f'<n>{i}</n>'.encode() for i in range(10)) + b'</r>')
total = 0
for _, elem in etree.iterparse(data, events=('end',), tag='n'):
    total += int(elem.text)
    elem.clear()
print(f"Somme: {total}")  # 45
```

</details>

### Exercice 5 *(difficile)* — Statistiques multi-critère

Parsez un XML de ventes avec `<sale><produit>X</produit><montant>N</montant></sale>`. Calculez le total et la moyenne par produit.

In [ ]:
from lxml import etree
import io
from collections import defaultdict

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree
import io
from collections import defaultdict

data = io.BytesIO(
    b'<ventes>'
    b'<sale><produit>A</produit><montant>10</montant></sale>'
    b'<sale><produit>B</produit><montant>20</montant></sale>'
    b'<sale><produit>A</produit><montant>5</montant></sale>'
    b'<sale><produit>B</produit><montant>30</montant></sale>'
    b'<sale><produit>A</produit><montant>15</montant></sale>'
    b'</ventes>'
)
totals: dict[str, list[float]] = defaultdict(list)
for event, elem in etree.iterparse(data, events=('end',), tag='sale'):
    p = elem.findtext('produit')
    m = float(elem.findtext('montant', '0'))
    totals[p].append(m)
    elem.clear()

for prod, vals in sorted(totals.items()):
    print(f"{prod}: total={sum(vals):.0f}, moyenne={sum(vals)/len(vals):.1f}, n={len(vals)}")
```

</details>

### Approfondir : comparaison de performance DOM vs iterparse

Pour les gros fichiers, la différence de mémoire entre DOM et iterparse
est spectaculaire. Voici un benchmark simplifié.

In [ ]:
import time
from lxml import etree
from io import BytesIO

# Génération d'un XML de test (5000 éléments)
xml_big = '<data>\n'
for i in range(5000):
    xml_big += f'  <item id="{i}"><value>{i*3.14:.2f}</value></item>\n'
xml_big += '</data>'
xml_bytes = xml_big.encode()

# DOM (parse tout en mémoire)
t0 = time.perf_counter()
tree = etree.parse(BytesIO(xml_bytes))
total_dom = sum(float(e.text) for e in tree.iter('value'))
t_dom = time.perf_counter() - t0

# iterparse (streaming)
t0 = time.perf_counter()
total_iter = 0
for event, elem in etree.iterparse(BytesIO(xml_bytes), events=('end',), tag='value'):
    total_iter += float(elem.text)
    elem.clear()
t_iter = time.perf_counter() - t0

print(f'DOM      : {t_dom*1000:.1f} ms, total={total_dom:.2f}')
print(f'iterparse: {t_iter*1000:.1f} ms, total={total_iter:.2f}')

### Exercice : agrégation multi-critères en streaming

Avec iterparse, calculez sur un XML de ventes :
1. Le chiffre d'affaires total par catégorie
2. Le nombre de ventes par mois
3. Le produit le plus vendu

Le tout sans charger le document en mémoire.

In [ ]:
from lxml import etree
from io import BytesIO
from collections import defaultdict
import random

# Génération du XML
random.seed(42)
categories = ['Electronique', 'Alimentaire', 'Textile', 'Mobilier']
produits = ['Laptop', 'Pain', 'T-shirt', 'Chaise', 'Phone', 'Lait', 'Jean', 'Table']
xml_ventes = '<ventes>\n'
for i in range(200):
    cat = random.choice(categories)
    prod = random.choice(produits)
    mois = f'2025-{random.randint(1,12):02d}'
    montant = round(random.uniform(5, 500), 2)
    xml_ventes += f'  <vente categorie="{cat}" mois="{mois}"><produit>{prod}</produit><montant>{montant}</montant></vente>\n'
xml_ventes += '</ventes>'

# Agrégation streaming
ca_par_cat = defaultdict(float)
ventes_par_mois = defaultdict(int)
ventes_par_produit = defaultdict(int)

for _, elem in etree.iterparse(BytesIO(xml_ventes.encode()), events=('end',), tag='vente'):
    cat = elem.get('categorie')
    mois = elem.get('mois')
    ca_par_cat[cat] += float(elem.findtext('montant'))
    ventes_par_mois[mois] += 1
    ventes_par_produit[elem.findtext('produit')] += 1
    elem.clear()

print('CA par catégorie :')
for cat, ca in sorted(ca_par_cat.items()):
    print(f'  {cat}: {ca:.2f} EUR')
print(f'\nProduit le plus vendu : {max(ventes_par_produit, key=ventes_par_produit.get)}')

### Bonnes pratiques pour le parsing XML volumineux

- **Toujours appeler `elem.clear()`** après traitement pour libérer la mémoire
- **Utiliser `tag='...'`** dans iterparse pour filtrer dès le parsing
- **Préférer `iterparse` à SAX** : API plus Pythonique, même performance
- **Pour les très gros fichiers** (> 1 Go) : envisager `iterparse` avec `lxml`
  plutôt que `xml.etree.ElementTree` (plus rapide en C)
- **Attention aux namespaces** : dans iterparse, le tag inclut le namespace
  complet `{http://...}tagname`

### Exercice bonus : détection de doublons en streaming

En utilisant iterparse, détectez les doublons (même identifiant)
dans un gros XML sans charger tout en mémoire.
Utilisez un set pour les IDs déjà vus.

In [ ]:
from lxml import etree
from io import BytesIO

# XML avec doublons
xml = '<items>\n'
import random; random.seed(42)
ids_used = []
for i in range(100):
    # 10% de chance de dupliquer un ID existant
    if ids_used and random.random() < 0.1:
        item_id = random.choice(ids_used)
    else:
        item_id = f'ITEM-{i:03d}'
        ids_used.append(item_id)
    xml += f'  <item id="{item_id}"><value>{i}</value></item>\n'
xml += '</items>'

# Détection streaming
seen = set()
duplicates = []
for _, elem in etree.iterparse(BytesIO(xml.encode()), events=('end',), tag='item'):
    item_id = elem.get('id')
    if item_id in seen:
        duplicates.append(item_id)
    else:
        seen.add(item_id)
    elem.clear()

print(f'{len(seen)} items uniques, {len(duplicates)} doublons')
print(f'IDs dupliqués : {duplicates}')

### Astuce : quand utiliser quoi ?

| Taille fichier | Méthode recommandée |
|---------------|--------------------|
| < 10 Mo | `etree.parse()` (DOM) |
| 10-100 Mo | `iterparse` avec `clear()` |
| > 100 Mo | `iterparse` avec `clear()` + tag filter |
| Streaming réseau | SAX `ContentHandler` |
| Besoin XPath | DOM obligatoire (ou découper) |
| Transformation | XSLT (via DOM) |

### Exercice : constructeur de résumé XML

Écrivez une fonction qui, via iterparse, crée un résumé structurel d'un XML :
- Liste des tags uniques avec leur fréquence
- Profondeur maximale
- Attributs rencontrés par tag

In [ ]:
from lxml import etree
from io import BytesIO
from collections import defaultdict

def xml_summary(xml_bytes):
    tag_counts = defaultdict(int)
    tag_attrs = defaultdict(set)
    depth = 0
    max_depth = 0

    for event, elem in etree.iterparse(
        BytesIO(xml_bytes), events=('start', 'end')
    ):
        if event == 'start':
            depth += 1
            max_depth = max(max_depth, depth)
            tag_counts[elem.tag] += 1
            for attr in elem.attrib:
                tag_attrs[elem.tag].add(attr)
        else:
            depth -= 1
            elem.clear()

    print(f'Profondeur max : {max_depth}')
    print(f'Tags uniques : {len(tag_counts)}')
    for tag, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
        attrs = tag_attrs[tag]
        attr_str = f' [{', '.join(attrs)}]' if attrs else ''
        print(f'  <{tag}>{attr_str} : {count}x')

# Test
xml_test = b'''
<company>
  <dept name="IT"><emp id="1"><name>Alice</name></emp><emp id="2"><name>Bob</name></emp></dept>
  <dept name="HR"><emp id="3"><name>Charlie</name></emp></dept>
</company>
'''
xml_summary(xml_test)

### Mémo : API iterparse

```python
# Événements disponibles
for event, elem in etree.iterparse(source, events=('start', 'end', 'start-ns', 'end-ns')):
    # 'start'    : ouverture du tag (attributs disponibles, pas le texte)
    # 'end'      : fermeture du tag (tout est disponible)
    # 'start-ns' : déclaration de namespace (elem = (prefix, uri))
    # 'end-ns'   : fin de portée du namespace
    pass
```

**Tip** : utilisez `tag='mon_tag'` pour filtrer et `elem.clear()` pour libérer la mémoire.

---

## Ressources externes

- [xml.sax — documentation Python](https://docs.python.org/3/library/xml.sax.html)
- [lxml iterparse](https://lxml.de/parsing.html#iterparse-and-iterwalk)
- [Processing large XML files](https://lxml.de/performance.html)